# Lesson 5: From project to package

This notebook goes with [Lesson 5 of *Unit Testing in Python*](https://acatlin.github.io/python-unit-testing/lessons/0005-from-project-to-package.html). The explanations are on that page. The code is here, ready to run.

- Run the cells from top to bottom, one at a time: click a cell and press Shift+Enter.
- This is your own copy. Change anything you like; nothing you do here alters the course. To keep your work, choose File, then Save a copy in Drive.

## How this notebook differs from the lesson page

On the page, step 3 makes a virtual environment, so that the package and its tools are kept apart from everything else on your computer. A Colab runtime already is one: a fresh machine that is yours alone and is thrown away when you disconnect. So this notebook skips step 3 and installs the package straight into the runtime. Switching an environment on would not last here in any case, because every `!` line starts a new shell.

On your own computer, do not skip it. The commands are at the end of this notebook.

## Set up

Colab comes with older versions of pandas and pytest than the lesson page was written with. The next cell installs the versions the page uses, so that what you see here matches it. It can take a minute.

pip may end with a message that other packages in Colab want the older pandas. This notebook does not use them, so carry on. If Colab asks you to restart the session, do that, then continue from the cell after this one.

In [ ]:
%pip install -q "pandas==3.1.1" "pytest==9.2.0"

Check what you have. A line that starts with `!` goes to the computer's command line, not to Python: it is this notebook's stand-in for the terminal.

In [ ]:
!python --version
!python -m pytest --version
!python -c "import pandas; print('pandas', pandas.__version__)"
!python -m pip --version

`pip install --group`, used below, needs pip 25.1 or newer. The last line above tells you which pip you have.

## How files and commands work here

- **`%%writefile name`** on the first line of a cell saves the rest of the cell as a file, in place of running it. To change the file, edit the cell and run it again.
- **`!`** at the start of a line sends it to the command line. Every `!` line starts in the notebook's own folder, so the commands below begin with `cd cleaner &&` to step into the package first.

**What will look different from the lesson page.** The page shows output captured on Windows with Python 3.14. Colab runs Linux with its own Python. So the `platform` line says `linux` and another Python version, `rootdir:` is `/content/cleaner`, paths are written with `/`, the timings differ, and pip's list of what it installed is shorter, because Colab has most of those packages already.

## Build it

### Step 1: make the folders

`src/cleaner/` for the code and `tests/` for the tests, then the files: `clean.py` as you finished it in Lesson 4 (only the first line, the description, has changed), an empty `__init__.py` next to it, and the test file.

In [ ]:
import os

os.makedirs("cleaner/src/cleaner", exist_ok=True)
os.makedirs("cleaner/tests", exist_ok=True)
open("cleaner/src/cleaner/__init__.py", "w").close()   # an empty file

In [ ]:
%%writefile cleaner/src/cleaner/clean.py
"""Data-cleaning helpers, now living in a package.
"""

import re
import warnings

import pandas as pd


def rescale01(x):
    """Rescale a numeric column so its smallest value is 0 and its largest is 1."""
    if not pd.api.types.is_numeric_dtype(x):
        raise TypeError("x must be a numeric column")
    lo, hi = x.min(), x.max()
    if lo == hi:
        return x * 0.0  # every value is the same: all zeros, missing kept
    return (x - lo) / (hi - lo)


def clean_names(names):
    """Turn messy column names into lower-case snake_case."""
    cleaned = []
    for name in names:
        name = re.sub(r"[^a-z0-9]+", "_", name.lower())
        cleaned.append(name.strip("_"))
    return cleaned


def parse_money(x):
    """Convert a column of strings such as "$1,200.50" to numbers."""
    digits = x.str.replace(r"[$,]", "", regex=True)
    numbers = pd.to_numeric(digits, errors="coerce").astype("float64")
    n_bad = int(numbers.isna().sum() - x.isna().sum())
    if n_bad > 0:
        warnings.warn(f"{n_bad} value(s) could not be parsed as money", stacklevel=2)
    return numbers


def summarise_by(data, group, value):
    """Count rows and average one column within each group."""
    return data.groupby(group, as_index=False).agg(
        n=(value, "size"),
        mean=(value, "mean"),
    )

The test file is the one from Lesson 4 with **one line changed** (step 5 on the page): the import now says `from cleaner.clean import …`, because the functions live in the module `clean` inside the package `cleaner`.

In [ ]:
%%writefile cleaner/tests/test_clean.py
import warnings

import pandas as pd
import pytest

from cleaner.clean import clean_names, parse_money, rescale01, summarise_by

# rescale01() ---------------------------------------------------------------


def test_rescale01_maps_smallest_to_0_and_largest_to_1():
    assert rescale01(pd.Series([0, 5, 10])).tolist() == [0, 0.5, 1]
    assert rescale01(pd.Series([-2, 0, 2])).tolist() == [0, 0.5, 1]


def test_rescale01_returns_one_float_per_input_value():
    x = rescale01(pd.Series([1, 2, 3]))
    assert x.dtype == "float64"
    assert len(x) == 3
    assert x.max() <= 1


def test_rescale01_keeps_missing_values_missing():
    result = rescale01(pd.Series([1, None, 3]))
    expected = pd.Series([0, None, 1])
    pd.testing.assert_series_equal(result, expected)


def test_rescale01_returns_zeros_when_every_value_is_the_same():
    assert rescale01(pd.Series([5, 5, 5])).tolist() == [0, 0, 0]


def test_rescale01_rejects_a_column_that_is_not_numeric():
    with pytest.raises(TypeError, match="must be a numeric column"):
        rescale01(pd.Series(["a", "b"]))


# clean_names() -------------------------------------------------------------


def test_clean_names_returns_lower_case_snake_case():
    assert clean_names(["First Name"]) == ["first_name"]
    assert clean_names(["Total ($)", " Zip-Code "]) == ["total", "zip_code"]


# parse_money() -------------------------------------------------------------


def test_parse_money_strips_dollar_signs_and_commas():
    assert parse_money(pd.Series(["$1,200.50"])).tolist() == [1200.5]
    assert parse_money(pd.Series(["75"])).tolist() == [75]


def test_parse_money_reads_thousands_without_cents():
    assert parse_money(pd.Series(["$3,000"])).tolist() == [3000]


def test_parse_money_warns_when_a_value_cannot_be_parsed():
    with pytest.warns(UserWarning, match="could not be parsed"):
        parse_money(pd.Series(["n/a"]))


def test_parse_money_is_quiet_when_every_value_parses():
    with warnings.catch_warnings():
        warnings.simplefilter("error")
        parse_money(pd.Series(["$5"]))


# summarise_by() ------------------------------------------------------------


def test_summarise_by_counts_rows_and_averages_within_each_group():
    staff = pd.DataFrame({
        "dept": ["a", "a", "b"],
        "pay": [10, 20, 60],
    })
    expected = pd.DataFrame({
        "dept": ["a", "b"],
        "n": [2, 1],
        "mean": [15.0, 60.0],
    })
    pd.testing.assert_frame_equal(summarise_by(staff, "dept", "pay"), expected)


def test_summarise_by_returns_a_data_frame_with_the_documented_columns():
    staff = pd.DataFrame({"dept": ["a", "a", "b"], "pay": [10, 20, 60]})
    out = summarise_by(staff, "dept", "pay")
    assert isinstance(out, pd.DataFrame)
    assert list(out.columns) == ["dept", "n", "mean"]

### Step 2: write the settings file

In [ ]:
%%writefile cleaner/pyproject.toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "cleaner"
version = "0.1.0"
description = "Small data-cleaning helpers"
requires-python = ">=3.11"
dependencies = ["pandas"]

[dependency-groups]
dev = ["pytest>=9"]

[tool.pytest]
testpaths = ["tests"]
addopts = ["--import-mode=importlib"]

Here is what you have made. Compare it with the tree on the lesson page.

In [ ]:
from pathlib import Path

for path in sorted(Path("cleaner").rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(path.as_posix())

### Step 3: the virtual environment

Skipped in Colab; see the top of this notebook.

### Step 4: install your package

In editable mode (`-e`), together with the `dev` group. The dot means "the project in this folder". Look for `Successfully installed cleaner-0.1.0` at the end.

In [ ]:
!cd cleaner && python -m pip install -e . --group dev

### Step 6: run the tests

Plain `pytest` is enough now. You should see `12 passed`.

In [ ]:
!cd cleaner && pytest

## What the editable install did

Python can now import your code from any folder. This line runs in the notebook's folder, not inside `cleaner/`.

In [ ]:
!python -c "from cleaner.clean import clean_names; print(clean_names(['First Name', ' Zip-Code ']))"

## The working loop

Only the tests whose names contain a word:

In [ ]:
!cd cleaner && pytest -k rescale01

Stop at the first failure (there is none yet):

In [ ]:
!cd cleaner && pytest -x

## Try it

Step 1 of the lesson is what you have just done.

2. Break `clean_names()` on purpose: in the cell that saves `clean.py`, delete `.lower()`, and run that cell. Nothing needs reinstalling, because the install is editable. Run `pytest` below and read the report. Run `pytest --lf` and notice how many tests it collects. Undo the change and run everything again.

In [ ]:
!cd cleaner && pytest

In [ ]:
!cd cleaner && pytest --lf

3. Add a new function to the package test-first: write a failing test, then make it pass. An idea: `squish()`, which trims a string and collapses runs of spaces into one. Put it in a new file with `%%writefile cleaner/src/cleaner/text.py`, and its tests in `%%writefile cleaner/tests/test_text.py`.

In [ ]:
# step 3: start with the test file

In [ ]:
# step 3: then the function

In [ ]:
!cd cleaner && pytest

## On your own computer

There, do step 3. In a terminal in the top `cleaner` folder:

**Windows**

```
python -m venv .venv
.venv\Scripts\activate
python -m pip install -e . --group dev
pytest
```

**macOS and Linux**

```
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e . --group dev
pytest
```

Now go back to the [lesson page](https://acatlin.github.io/python-unit-testing/lessons/0005-from-project-to-package.html) for the quiz.